In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from mlxtend.plotting import plot_confusion_matrix
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score


ModuleNotFoundError: No module named 'mlxtend'

In [ ]:
df = pd.read_csv('iris.csv')

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
df.hist(figsize=(10, 10))

In [ ]:
df.head(100)

Filter only those records where flower type is setosa or versicolor. [Y Column should be numeric say 0 or 1]

In [ ]:
filtered_df = df[df['Species'].isin(['Iris-setosa', 'Iris-versicolor'])].copy()
filtered_df['Species'] = filtered_df['Species'].map({'Iris-setosa': 0, 'Iris-versicolor': 1})
features = filtered_df[['SepalLengthCm', 'SepalWidthCm']]
X = features.values
Y = filtered_df['Species'].to_numpy(dtype=np.int64)

## Normalization of X

In [ ]:
def normalize(X):
    print("Mean Before: ", X.mean(axis=0), "\nStd dev. Before: ", X.std(axis=0))

    sc = StandardScaler()
    XScaled = sc.fit_transform(X)

    print("Mean After: ",XScaled.mean(axis=0), "\nStd dev. After: ",XScaled.std(axis=0))
    return XScaled

## Train-Test and Split using K-fold cross validation

In [ ]:

def LogisticReg(X, y, fold_scores):
    kf = KFold(n_splits=10, shuffle=True, random_state=42)
    for train_index, test_index in kf.split(X):
        # Split data into train and test sets for the current fold
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]
        LRModel = LogisticRegression()
        LRModel.fit(X_train, y_train)
        score = LRModel.score(X_test, y_test)
        fold_scores.append(score)

        Ytest_pred = LRModel.predict(X_test)
        Ytrain_pred = LRModel.predict(X_train)
        print("Weight: ", LRModel.coef_)
        print("Intercept: ", LRModel.intercept_)
        print("Predicted labels: ", Ytest_pred)
        print("Training Accuracy: ", accuracy_score(y_train, Ytrain_pred))
        print("Test Accuracy: ", accuracy_score(y_test, Ytest_pred))

        confusion_matrix(y_test, Ytest_pred)
        plot_confusion_matrix(confusion_matrix(y_test, Ytest_pred), figsize=(6, 6), hide_ticks=True, cmap=plt.cm.Blues)
        print(classification_report(y_test, Ytest_pred))
        roc_curve(y_test, Ytest_pred)
        print("ROC AUC Score: ", roc_auc_score(y_test, Ytest_pred))
        return fold_scores


## Create a Logistic Regression model by training on train partition(X_train, Y_train).

In [ ]:
XScaled = normalize(X)
result = LogisticReg(XScaled, Y, [])
result

In [ ]:
print(type(Y), Y.shape)
print('dtype:', Y.dtype)
print('first 10:', Y[:10])
print('unique:', np.unique(Y))